# Opportunity-Adjusted Officiating Analysis

A companion notebook to the Argentina-Egypt comeback case study. It studies whether penalties, fouls, and cards track measurable attacking opportunity.

## tl;dr

- The official 2026 FIFA panel covers **96 matches / 192 team-match rows**. Argentina received **3 penalties in 5 matches**, but the tournament contains only 19 official awards, so it supports descriptive association rather than a large adjusted model.
- In the PMSR 2026 panel, the match-clustered Spearman association between xG and penalties awarded is **0.380** (95% bootstrap interval 0.281 to 0.469).
- The 2018/2022 StatsBomb World Cup panel covers **128 matches / 256 team-match rows**. Its association between box entries and regulation penalty kicks is **0.026** (95% bootstrap interval -0.101 to 0.140).
- The time-aware club panel covers **2193 matches and 604 penalty-shot events**. Its grouped out-of-fold hazard model has log loss **0.0294** versus **0.0309** for the null rate.
- The linked club bridge model compares final-result log loss **0.7895** for the state-only model with **0.7847** after adding pre-window opportunity features.
- These are opportunity-adjusted associations, not an adjudication of referee decisions. The initial model does not include pre-decision score state, strength, referee effects, or a complete denominator of potential box incidents.

## Context & Methods

### Key Assumptions

The data lanes are never pooled. FIFA PMSR 2026 measures official penalties awarded; StatsBomb uses regulation penalty kicks as an event-feed proxy. The principal opportunity features are spatial box entries and non-penalty xG, not possession alone. Statistics are cluster-bootstrapped by match, and the small Poisson model is evaluated out-of-fold by complete match.

See `docs/opportunity_adjusted_analysis.md` for the full estimand, source, and publication contract.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

ROOT = Path.cwd()
while not (ROOT / 'Project.md').exists():
    if ROOT.parent == ROOT:
        raise RuntimeError('Run from this repository or a subdirectory.')
    ROOT = ROOT.parent

PROCESSED = ROOT / 'data' / 'processed'
RESULTS = ROOT / 'results'
manifest = json.loads((RESULTS / 'opportunity_analysis_manifest.json').read_text(encoding='utf-8'))
print(manifest['interpretation'])
print('Non-pooling rule:', manifest['non_pooling_rule'])

## Data

### Load the Official and Event-Level Panels

In [ ]:
pmsr = pd.read_csv(PROCESSED / 'pmsr_2026_match_features.csv')
world_cup = pd.read_csv(PROCESSED / 'statsbomb_world_cup_opportunity_features.csv')
assert len(pmsr) == 192 and pmsr['match_number'].nunique() == 96
assert len(world_cup) == 256 and world_cup['match_id'].nunique() == 128
display(pmsr[['match_number', 'team_name', 'opponent_team_name', 'penalties_awarded', 'team_possession_pct', 'xg', 'receptions_final_third']].head(8))
display(world_cup[['year', 'team_name', 'opponent_team_name', 'penalty_shots', 'box_entries', 'box_receipts', 'non_penalty_xg', 'fouls_committed', 'yellow_cards']].head(8))

### Outcome Definition Check

PMSR's `penalties_awarded` is the official count. StatsBomb's `penalty_shots` is used separately because its `Foul Won -> penalty` flag does not capture every eventual penalty kick in the feed. The two fields are therefore never merged or compared as if they were the same measurement.

In [ ]:
reconciliation = world_cup.groupby('year')[['penalty_fouls_won', 'penalty_shots']].sum()
reconciliation['difference'] = reconciliation['penalty_shots'] - reconciliation['penalty_fouls_won']
display(reconciliation)

### Formal Correlation Inference

The correlation table adds a 95% match-cluster bootstrap interval and a whole-match permutation p-value. The permutation keeps the two team rows together while breaking the alignment between exposure and outcome clusters. Benjamini-Hochberg q-values control the registered correlation family. Significance is evidence of association under this design, not causality or officiating bias.

In [ ]:
correlation_path = RESULTS / 'opportunity_correlation_tests.csv'
if correlation_path.exists():
    correlation_tests = pd.read_csv(correlation_path)
    display(correlation_tests)
else:
    print('Formal correlation tests have not been generated in this run.')

## Results

### Official 2026 Penalties and Attacking Output

In [ ]:
pmsr_summary = pd.read_csv(RESULTS / 'opportunity_pmsr_2026_team_summary.csv')
display(pmsr_summary.head(12))
display(Image(filename=str(RESULTS / 'assets' / '09_pmsr_2026_penalties_vs_xg.png')))

The scatter is a descriptive diagnostic: xG is a better proxy for attacking threat than total possession, but the small number of penalty awards prevents a stable multi-covariate 2026-only model.

### World Cup 2018/2022: Box Access and Penalty Kicks

In [ ]:
world_predictions = pd.read_csv(RESULTS / 'opportunity_statsbomb_world_cup_oof_predictions.csv')
world_model = next(item for item in manifest['models'] if item['panel'] == 'statsbomb_world_cup')
if world_model['oof_beats_null']:
    display(pd.read_csv(RESULTS / 'opportunity_statsbomb_world_cup_team_residuals.csv').head(12))
else:
    print('The World Cup partial model did not beat its null baseline out-of-fold; residual ranks are withheld.')
display(Image(filename=str(RESULTS / 'assets' / '10_statsbomb_world_cup_box_entries_vs_penalty_shots.png')))

The residual table is an out-of-fold partial adjustment using only box entries and non-penalty xG. It is useful for identifying where raw penalty counts are not fully explained by those two features, but it is not a decision-correctness or bias score.

### Optional Broader Baseline

In [ ]:
history_path = PROCESSED / 'statsbomb_history_opportunity_features.csv'
if history_path.exists():
    history = pd.read_csv(history_path)
    print(history.groupby('competition_tier')['match_id'].nunique())
    for tier in sorted(history['competition_tier'].unique()):
        residual_path = RESULTS / f'opportunity_statsbomb_{tier}_team_residuals.csv'
        if residual_path.exists():
            print(f'\n{tier}')
            display(pd.read_csv(residual_path).head(10))
else:
    print('Broader StatsBomb feature panel has not been generated in this run.')

## Time-Aware Pre-Decision Hazard

The temporal extension creates one row for each team and five-minute window. Every predictor is calculated strictly before that window: score state, recent box entries, recent xG, fouls, cards, and event-derived possession sequences. The next-window label is whether a StatsBomb penalty shot occurs. This prevents using the event being explained as its own predictor. Panels remain separate by competition tier, and the model is evaluated out-of-fold by complete match.

In [ ]:
temporal_manifest_path = RESULTS / 'temporal_analysis_manifest.json'
if temporal_manifest_path.exists():
    temporal_manifest = json.loads(temporal_manifest_path.read_text(encoding='utf-8'))
    temporal_rows = []
    for item in temporal_manifest['models']:
        temporal_rows.append({
            'panel': item['panel'], 'tier': item['tier'], 'matches': item['matches'],
            'five_minute_rows': item['rows'], 'penalty_events': item['positive_events'],
            'positive_rate': item['positive_rate'], 'oof_log_loss': item['oof_log_loss'],
            'null_log_loss': item['null_log_loss'], 'oof_beats_null': item['oof_beats_null'],
        })
    display(pd.DataFrame(temporal_rows))
    club_predictions = pd.read_csv(RESULTS / 'temporal_statsbomb_club_oof_predictions.csv')
    club_coefficients = pd.read_csv(RESULTS / 'temporal_statsbomb_club_coefficients.csv')
    display(club_coefficients)
    display(Image(filename=str(RESULTS / 'assets' / '11_temporal_club_penalty_hazard.png')))
else:
    print('Temporal panel has not been generated in this run.')

The hazard result is a mechanism check, not a bias estimate. A model that slightly beats the null means that pre-window match context contains information about where penalty-shot events occur; it does not establish that the underlying decision was correct, incorrect, biased, or intentional.

## Bridge to Notebook 1: Outcome Probability and Penalty Impact

This section uses the same match-state logic as Notebook 1: predict the eventual regulation result from the state at a given time, keeping complete matches together in every validation fold. It compares a state-only model with spatial, possession, disciplinary, and opponent-opportunity extensions. The primary counterfactual changes only `goal_diff_pre` by one goal, representing an immediate penalty goal for the team or for its opponent. It is a score-state sensitivity estimate, not a full tactical simulation and not a judgment about whether a penalty should have been awarded.

In [ ]:
bridge_manifest_path = RESULTS / 'bridge_analysis_manifest.json'
if bridge_manifest_path.exists():
    bridge_manifest = json.loads(bridge_manifest_path.read_text(encoding='utf-8'))
    bridge_skill = pd.read_csv(RESULTS / 'bridge_model_skill.csv')
    bridge_impact = pd.read_csv(RESULTS / 'bridge_penalty_win_impact.csv')
    display(bridge_skill[['panel', 'tier', 'variant', 'rows', 'log_loss', 'multiclass_brier', 'accuracy']])
    display(bridge_impact.loc[bridge_impact['cohort'].isin(['late_trailing', 'late_two_or_more_down'])].head(20))
    display(Image(filename=str(RESULTS / 'assets' / '12_bridge_model_skill.png')))
    impact_chart = RESULTS / 'assets' / '13_penalty_win_impact.png'
    if impact_chart.exists():
        display(Image(filename=str(impact_chart)))
else:
    print('Bridge analysis has not been generated in this run.')

The bridge is a linked robustness analysis, not a replacement for Notebook 1's validated World Cup model. The StatsBomb tiers have different competition populations, and the 2026 focal match does not yet have the same pre-window event-level covariates. Therefore the bridge results validate mechanism and sensitivity, not a new focal probability.

## Takeaways

- Possession should be interpreted alongside location and chance-quality measures. A team can control the ball away from the opponent box, while a lower-possession team can still create many high-leverage entries.
- Club, non-World-Cup international, World Cup, and 2026 PMSR findings are linked by common metric definitions but remain separate estimands.
- The time-aware panel now adds score state and recent spatial/event exposure without leaking the outcome window. The next methodological upgrade is a validated denominator of potential penalty incidents and stronger national-team coverage.